In [0]:
display(spark.sql("SHOW TABLES IN samples.tpch"))

**Imports e Paths**

In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name

volume_path = "/Volumes/erp_lakehouse/landing/raw_files"
checkpoint_base = "/Volumes/erp_lakehouse/landing/pipeline_metadata/_checkpoints"
schema_base = "/Volumes/erp_lakehouse/landing/pipeline_metadata/_schemas"

spark.sql("CREATE SCHEMA IF NOT EXISTS erp_lakehouse.bronze")

tabelas = ["customer", "orders", "lineitem", "part", "supplier"]

**Função Reutilizável de Ingestão**

In [0]:
from pyspark.sql.functions import current_timestamp, col

def ingest_bronze(tabela: str):
    df = (spark.readStream
          .format("cloudFiles")
          .option("cloudFiles.format", "csv")
          .option("header", "true")
          .option("cloudFiles.schemaLocation", f"{schema_base}/{tabela}")
          .option("cloudFiles.inferColumnTypes", "true")
          .load(f"{volume_path}/{tabela}")
          .withColumn("_ingested_at", current_timestamp())
          .withColumn("_source_file", col("_metadata.file_path")))

    query = (df.writeStream
             .format("delta")
             .option("checkpointLocation", f"{checkpoint_base}/{tabela}")
             .trigger(availableNow=True)
             .toTable(f"erp_lakehouse.bronze.{tabela}"))

    query.awaitTermination()
    print(f"✅ {tabela} ingerida na Bronze")

**Executar para todas as tabelas**

In [0]:
for tabela in tabelas:
    ingest_bronze(tabela)